## Lesson 6 – Loan risk data and cleaning

### Step 1: Load the data and its dictionary

In [4]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

credit = fetch_ucirepo(id=350)
X = credit.data.features
y = credit.data.targets

print(X.shape)
print(y.value_counts())
print(credit.variables[["name", "description"]].to_string())

(30000, 23)
Y
0    23364
1     6636
Name: count, dtype: int64
   name                 description
0    ID                         NaN
1    X1                   LIMIT_BAL
2    X2                         SEX
3    X3                   EDUCATION
4    X4                    MARRIAGE
5    X5                         AGE
6    X6                       PAY_0
7    X7                       PAY_2
8    X8                       PAY_3
9    X9                       PAY_4
10  X10                       PAY_5
11  X11                       PAY_6
12  X12                   BILL_AMT1
13  X13                   BILL_AMT2
14  X14                   BILL_AMT3
15  X15                   BILL_AMT4
16  X16                   BILL_AMT5
17  X17                   BILL_AMT6
18  X18                    PAY_AMT1
19  X19                    PAY_AMT2
20  X20                    PAY_AMT3
21  X21                    PAY_AMT4
22  X22                    PAY_AMT5
23  X23                    PAY_AMT6
24    Y  default payment next month


### Step 2: Printing the metadata 
know more about what those columns represent

In [5]:
print(credit.metadata.additional_info.variable_info)

This research employed a binary variable, default payment (Yes = 1, No = 0), as the response variable. This study reviewed the literature and used the following 23 variables as explanatory variables:
X1: Amount of the given credit (NT dollar): it includes both the individual consumer credit and his/her family (supplementary) credit.
X2: Gender (1 = male; 2 = female).
X3: Education (1 = graduate school; 2 = university; 3 = high school; 4 = others).
X4: Marital status (1 = married; 2 = single; 3 = others).
X5: Age (year).
X6 - X11: History of past payment. We tracked the past monthly payment records (from April to September, 2005) as follows: X6 = the repayment status in September, 2005; X7 = the repayment status in August, 2005; . . .;X11 = the repayment status in April, 2005. The measurement scale for the repayment status is: -1 = pay duly; 1 = payment delay for one month; 2 = payment delay for two months; . . .; 8 = payment delay for eight months; 9 = payment delay for nine months and

### Step 3: Rename the columns

In [6]:
X.columns = [
    "credit_limit", "sex", "education", "marriage", "age",
    "status_sep", "status_aug", "status_jul", "status_jun", "status_may", "status_apr",
    "bill_sep", "bill_aug", "bill_jul", "bill_jun", "bill_may", "bill_apr",
    "paid_sep", "paid_aug", "paid_jul", "paid_jun", "paid_may", "paid_apr",
]
y = y["Y"].rename("defaulted")

print(X.head())

   credit_limit  sex  education  marriage  age  status_sep  status_aug  \
0         20000    2          2         1   24           2           2   
1        120000    2          2         2   26          -1           2   
2         90000    2          2         2   34           0           0   
3         50000    2          2         1   37           0           0   
4         50000    1          2         1   57          -1           0   

   status_jul  status_jun  status_may  ...  bill_jul  bill_jun  bill_may  \
0          -1          -1          -2  ...       689         0         0   
1           0           0           0  ...      2682      3272      3455   
2           0           0           0  ...     13559     14331     14948   
3           0           0           0  ...     49291     28314     28959   
4          -1           0           0  ...     35835     20940     19146   

   bill_apr  paid_sep  paid_aug  paid_jul  paid_jun  paid_may  paid_apr  
0         0         0   

### Step 4: Check every code against the dictionary

In [7]:
for col in ["education", "marriage", "status_sep"]:
    print(col)
    print(X[col].value_counts().sort_index())
    print()

education
education
0       14
1    10585
2    14030
3     4917
4      123
5      280
6       51
Name: count, dtype: int64

marriage
marriage
0       54
1    13659
2    15964
3      323
Name: count, dtype: int64

status_sep
status_sep
-2     2759
-1     5686
 0    14737
 1     3688
 2     2667
 3      322
 4       76
 5       26
 6       11
 7        9
 8       19
Name: count, dtype: int64



### Step 5: Test what -2 means

In [8]:
no_bill = X["bill_sep"] <= 0

for code in [-2, -1, 0, 1]:
    group = X["status_sep"] == code
    print(f"status {code}: {no_bill[group].mean():.0%} had no bill in September")

status -2: 33% had no bill in September
status -1: 0% had no bill in September
status 0: 0% had no bill in September
status 1: 46% had no bill in September


### Step 6: Default rate by status code

In [9]:
print(pd.crosstab(X["status_sep"], y, normalize="index").round(2))

defaulted      0     1
status_sep            
-2          0.87  0.13
-1          0.83  0.17
 0          0.87  0.13
 1          0.66  0.34
 2          0.31  0.69
 3          0.24  0.76
 4          0.32  0.68
 5          0.50  0.50
 6          0.45  0.55
 7          0.22  0.78
 8          0.42  0.58


### Step 7: Apply the cleaning and save it

In [10]:
import os

X_clean = X.copy()
X_clean["education"] = X_clean["education"].replace({0: 4, 5: 4, 6: 4})
X_clean["marriage"] = X_clean["marriage"].replace({0: 3})

print(X_clean["education"].value_counts().sort_index())
print(X_clean["marriage"].value_counts().sort_index())

os.makedirs("data", exist_ok=True)
X_clean.assign(defaulted=y).to_csv("data/credit_clean.csv", index=False)
print("Saved:", X_clean.shape)

education
1    10585
2    14030
3     4917
4      468
Name: count, dtype: int64
marriage
1    13659
2    15964
3      377
Name: count, dtype: int64
Saved: (30000, 23)


### Lesson 6 summary – Loan risk data and cleaning

**The data:** 30,000 credit card customers from a bank in Taiwan (2005). 22% defaulted the next month, compared with 12% "yes" in the deposit data.

**What I did**
```
1. LOAD      → 30,000 customers, 23 columns + default answer
2. RENAME    → X1–X23 became readable names (credit_limit, status_sep, bill_sep…)
3. CHECK     → compared every code with the data dictionary
4. TEST      → checked what the unclear status codes really mean
5. DECIDE    → chose a fix based on the size of each problem
6. SAVE      → data/credit_clean.csv
```

**What didn't match the dictionary**

| Column | Allowed | Found | Customers | Decision |
|---|---|---|---|---|
| education | 1–4 | 0, 5, 6 | 345 (~1%) | Merged into 4 ("other") |
| marriage | 1–3 | 0 | 54 (~0.2%) | Merged into 3 ("other") |
| status | -1, 1–9 | -2, 0 | 17,496 (58%) | Kept as is |

**Testing the status codes**
- My guess "-2 means no bill" was wrong: only 33% of -2 customers had no bill.
- Better question: how risky is each code?
  - -2, -1, 0 → 13–17% default (paying normally)
  - 1 → 34%, 2 and up → around 70% (behind on payments)
- The codes are in a sensible order of risk, so they stay. Deleting 58% of customers would throw away the most useful column.
- Codes 4–8 have very few customers, so their rates are unreliable.

**Key lessons**
- Don't trust the data dictionary blindly.
- Test guesses with the data instead of assuming.
- Don't delete what you don't understand.
- When labels are unclear, check behaviour: what matters is how the codes relate to default.
- The size of the problem decides the fix.

**How I'd explain it to a manager:** "Some codes in the data weren’t in the documentation. For education and marriage, they affected only about 1% of customers and carried no meaning, so I grouped them under ‘other’ without losing anyone. The unexplained payment-status codes covered 58% of customers, and when I tested them they clearly tracked default risk: customers marked -2, -1 or 0 defaulted 13–17% of the time, versus 34–78% for those behind on payments. So I kept them, because removing them would have thrown away our strongest risk signal."